### 시나리오
 - AI관련 교육상담을 하는 챗봇을 구성한다.
 - 최근 정부에서 'AI캠퍼스'라는 교육 사업을 추진 중
 - 해당 사업에 대한 공고문(pdf)
   - https://smhrd.or.kr/course/aic_network/

### 학습내용
 - Multi-Source RAG 실습 (PDF와 웹페이지 retriever 정보 결합)
 - Query Transformation 실습
   - ex) AI캠퍼스 교육과정 어때? => 사용자는 보통 질문을 축약해서 하는 편
   - LLM에 변경요청 => AI캠퍼스의 사업목표, 교육 과정 종류, 시수, 커리큘럼 등을 알려줘

In [ ]:
# Open AI API key 등록
import os

OPENAI_API_KEY='API_KEY'

# 현재 노트북 커널에 환경변수 등록
os.environ['OPENAI_API_KEY']=OPENAI_API_KEY

# 랭스미스로 기록 할 건지 여부
LANGSMITH_TRACING="true"

# 랭스미스 사이트에서 기록 할 나만의 url
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"

# 내 랭스미스 url key
LANGSMITH_API_KEY="API_KEY"

# 내 계정의 프로젝트이름
LANGSMITH_PROJECT="langchain0422"

# 현재 노트북 커널에 환경변수 등록
os.environ['LANGSMITH_TRACING']=LANGSMITH_TRACING
os.environ['LANGSMITH_ENDPOINT']=LANGSMITH_ENDPOINT
os.environ['LANGSMITH_API_KEY']=LANGSMITH_API_KEY
os.environ['LANGSMITH_PROJECT']=LANGSMITH_PROJECT

In [22]:
# 라이브러리 설치하기

!pip install langchain langchain-openai langchain-community faiss-cpu pypdf chromadb

### 1. Query Transformation

In [108]:
# ** RAG 관련 도구 **

#
from langchain_community.document_loaders import PyPDFLoader

# 재귀적 텍스트 분할 도구
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 임베딩 도구
from langchain_openai import OpenAIEmbeddings

# 벡터 데이터 베이스 도구
from langchain_community.vectorstores import FAISS

# 문서 도구
from langchain_core.documents import Document


# ** Chain 구성을 위한 도구 **

# 프롬프트 템플릿
from langchain_core.prompts import ChatPromptTemplate

# LLM 생성하는 도구
from langchain.chat_models import init_chat_model

# 출력파서 도구
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

# 구조 생성용 Runnable 도구
from langchain_core.runnables import RunnablePassthrough, RunnableBranch, RunnableParallel, RunnableLambda

# 문자열 구성 도구
from langchain_core.prompts import PromptTemplate

In [24]:
QT_template = ChatPromptTemplate.from_template('''
    너는 사용자의 query를 retriever가 더 잘 검색하도록 최적화화는 역할이야.

    - 수행내용
        1. 아래 query를 기반으로 사용자의 질문 의도를 추출
        2. 사용자 질문을 깔끔하고 정도된 질문으로 변환
        3. 사용자의 질문을 다양한 관점으로 확장해서 3가지 정도로 늘리기

    - query : {query}

    - 출력형식
        아래 key 값을 이용해서 파싱가능한 JSON 구조로 만들어줘

        key :
            - user_intent
            - normalize_query
            - expanded_query
''')

In [25]:
# 체인구성하기
llm_4o_mini = init_chat_model('openai:gpt-4o-mini', max_tokens=1024)
jsonParser = JsonOutputParser()

QT_chain = QT_template | llm_4o_mini | jsonParser

In [26]:
# q1
QT_chain.invoke("AI교육 추천해줘")

{'user_intent': 'AI 교육 프로그램이나 자료를 추천받고 싶다.',
 'normalize_query': 'AI 교육 프로그램을 추천해 주세요.',
 'expanded_query': ['어떤 AI 관련 교육 과정이 가장 효과적인가요?',
  '현재 추천할 만한 AI 교육 자료가 있나요?',
  'AI 교육을 위한 온라인 플랫폼이나 기관을 추천해 주세요.']}

### 2. 파일기반 retriever 구성하기

In [27]:
# 파일로더 생성
loader = PyPDFLoader('/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM활용/data/260107 K-디지털 트레이닝 AI 캠퍼스 운영(인적자원개발과).pdf')
# 데이터로딩
documents = loader.load()

In [28]:
type(documents)

list

In [29]:
type(documents[0])

langchain_core.documents.base.Document

In [30]:
pdf_splitter = RecursiveCharacterTextSplitter(
    separators = ['\n\n', '\n', ' ', ''],   # separators 생략시 문단, 줄바꿈, 띄어쓰기, 글자 순으로 자르도록 기본 설정 됨
    chunk_size = 500,   # 청크의 글자 크기
    chunk_overlap = 100   # 청크 오버랩 크기
)

In [31]:
pdf_chunk = pdf_splitter.split_documents(documents)

In [32]:
len(pdf_chunk)

12

In [34]:
# 임베딩 및 인덱싱
vec_db = FAISS.from_documents(
    documents = pdf_chunk,   # 임베딩할 벡터
    embedding = OpenAIEmbeddings()   # 임베딩 도구 연결
)

In [37]:
# 사용자 질문과 유사한 청크 2개를 찾는 검색기 생성
pdf_retriever = vec_db.as_retriever(search_kwargs = {'k' : 2})

In [38]:
# 테스트
pdf_retriever.invoke('AI캠퍼스 과정은 얼마나 어려운가요?')

[Document(id='7d8c9c39-38e4-4792-a8b2-85eec90dbdcb', metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2022 12.0.0.4204', 'creationdate': '2026-01-06T14:12:03+09:00', 'author': 'Moel', 'moddate': '2026-01-06T14:12:03+09:00', 'pdfversion': '1.4', 'source': '/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM활용/data/260107 K-디지털 트레이닝 AI 캠퍼스 운영(인적자원개발과).pdf', 'total_pages': 4, 'page': 3, 'page_label': '4'}, page_content='붙임 AI 캠퍼스 양성 목표 직종(직무) 직군분류 (예시 직종/직무) 직무정의① AI 엔지니어(AI Engineer)"어 떻 게  구 현 하 는 가?" 연구자가 밝혀낸 원리를 바탕으로, 실 제  사 용 자 가  쓸 수 있는 안정적인 서비스를 구축하고 운영'),
 Document(id='f497bd80-7136-4a0f-8bf0-f08ab4db3c51', metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2022 12.0.0.4204', 'creationdate': '2026-01-06T14:12:03+09:00', 'author': 'Moel', 'moddate': '2026-01-06T14:12:03+09:00', 'pdfversion': '1.4', 'source': '/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM활용/data/260107 K-디지털 트레이닝 AI 캠퍼스 운영(인적자원

### 3. 웹페이지기반 retriever 구성하기
 - 기본 WebBaseLoader는 정적페이지는 잘 불러옴
 - 동적페이지를 불러오려면 외부 도구나 selenium을 활용하는것이 효과적이다.

In [39]:
# 앞쪽에서 selenium을 통해 웹문서의 text를 추출했다는 가정하에 진행
text1 = '''
    6개월 과정960시간오프라인
파이썬과 AI수학
- 파이썬 분석/시각화, 선형대수, 확률, 통계

컴퓨터 비전
- 영상처리, 객체탐지/인식, 동작인식, 얼굴인식, 영역분할

머신러닝/딥러닝
- 지도/비지도학습, 평가지표, 신경망, CNN, RNN, Transformer, LMM/VLM/VLA

로봇 운동학
- 3차원공간 이해, 좌표계 변환, 자유도, FK/IK 해석, 로봇 움직임 제어, PID

ROS2과 SLAM
- ROS2, 거리 및 IMU 센서, 점군 데이터 처리, SLAM, Navigation

가상 로봇 설계
- URDF/SDF기반 설계, 가상센서, 가상 로봇 제어

가상 로봇 제어
- 맵핑, 장애물 회피, 복구행동, 경로계획, 행동결정, 로봇팔, 다중로봇, 자율주행

현실 로봇 제어
- ROS brigde, 카메라/Lidar 연동, 모터 제어, 자율주행

배포 서비스 구현
- Streamlit/FastAPI

AI 서비스 설계
- AI 웹 서비스 기술 (HTML/CSS, JS, Node.js. React.js)

'''

In [40]:
doc1 = Document(
    page_content = text1,
    metadata = {'source' : 'https://smhrd.or.kr/course/aic_network/'}
)

In [41]:
doc1

Document(metadata={'source': 'https://smhrd.or.kr/course/aic_network/'}, page_content='\n    6개월 과정960시간오프라인\n파이썬과 AI수학\n- 파이썬 분석/시각화, 선형대수, 확률, 통계\n\n컴퓨터 비전\n- 영상처리, 객체탐지/인식, 동작인식, 얼굴인식, 영역분할\n\n머신러닝/딥러닝\n- 지도/비지도학습, 평가지표, 신경망, CNN, RNN, Transformer, LMM/VLM/VLA\n\n로봇 운동학\n- 3차원공간 이해, 좌표계 변환, 자유도, FK/IK 해석, 로봇 움직임 제어, PID\n\nROS2과 SLAM\n- ROS2, 거리 및 IMU 센서, 점군 데이터 처리, SLAM, Navigation\n\n가상 로봇 설계\n- URDF/SDF기반 설계, 가상센서, 가상 로봇 제어\n\n가상 로봇 제어\n- 맵핑, 장애물 회피, 복구행동, 경로계획, 행동결정, 로봇팔, 다중로봇, 자율주행\n\n현실 로봇 제어\n- ROS brigde, 카메라/Lidar 연동, 모터 제어, 자율주행\n\n배포 서비스 구현\n- Streamlit/FastAPI\n\nAI 서비스 설계\n- AI 웹 서비스 기술 (HTML/CSS, JS, Node.js. React.js)\n\n')

In [45]:
text2 = '''
    6개월 과정960시간오프라인
파이썬과 AI수학
- 파이썬 분석/시각화, 선형대수, 확률, 통계

데이터 수집/전처리
- 크롤링, 데이터 구조 (JSON, XML, CSV 등)

머신러닝/딥러닝
- 지도/비지도학습, 평가지표, 신경망, CNN, RNN, Transformer, LMM/VLM/VLA

컴퓨터 비전
- 영상처리, 객체탐지/인식, 동작인식, 얼굴인식, 영역분할

텍스트마이닝
- 텍스트 전처리, 임베딩, 기본 모델 실습, Transformer, 모델 활용 실습

음성데이터 / 시계열 데이터
- 텍스트 전처리, 임베딩, 기본 모델 실습, Transformer, 모델 활용 실습

LangChain
- 프롬프트, 템플릿, 체인 생성/연결, 메모리, RAG, 고급 RAG

AI Agent
- LangServe, LangSmith, Ollama, LangGraph

Vision Agent
- 종류, 구현실습, LangGraph 기반 구현

배포 서비스
- Streamlit / FastAPI / n8n

AI 서비스 설계
- AI 웹 서비스 기술(HTML/CSS, JS, Node.js, React.js)

'''

In [46]:
doc2 = Document(
    page_content = text1,
    metadata = {'source' : 'https://smhrd.or.kr/course/aic_network/'}
)

In [47]:
text3 = '''
   6개월 과정960시간오프라인
헬스케어 AI와 파이썬
- 헬스케어 산업/기술/현황, 파이썬 분석/시각화, 선형대수, 확률, 통계

Java
- 정보공학, 환경설정, 연산자, 제어문, 배열, OOP, 예외처리, I/O, 쓰레드

JSP/Spring
- HTML, CSS, JS, JSP, Spring, Spring AI

데이터베이스
- 데이터베이스 개론, RDBMS실습, NoSQL

데이터 수집/전처리
- 헬스케어 데이터 크롤링, 데이터 구조 (JSON, XML, CSV 등)

머신러닝/딥러닝
- 지도/비지도학습, 평가지표, 신경망, CNN, RNN, Transformer, LMM/VLM/VLA

컴퓨터 비전
- 영상처리, 객체탐지/인식, 동작인식, 얼굴인식, 영역분할

텍스트마이닝
- 텍스트 전처리, 임베딩, 기본 모델 실습, Transformer, 모델 활용 실습

LangChain
- 프롬프트, 템플릿, 체인 생성 /연결, 메모리, RAG, 고급 RAG

n8n
- 노드, 워크플로우 설계, 인증, AI 자동화, MCP
'''

In [48]:
doc3 = Document(
    page_content = text1,
    metadata = {'source' : 'https://smhrd.or.kr/course/aic_network/'}
)

In [49]:
text4 = '''
   6개월 과정960시간오프라인
지역산업 AI 비즈니스 이해
- 지역특화산업/현황, AI 서비스 기획, 비즈니스브리프, 고객 여정맵, 시장조사, KPI

고객경험 도메인 데이터 설계
- 경험데이터 분석, DCX개념에 따른 고객맞춤형 서비스 모델 이해, 아이디어 토론 및 공유

X+AI와 파이썬
- 지역도메인AI현황 분석, 파이썬 분석/시각화, 선형대수, 확률, 통계

데이터베이스
- 데이터베이스 개론, RDBMS실습, NoSQL

통계적분석기반 도메인 데이터 이해
- 실데이터 분석, 시나리오 정의, 인사이트 도출 및 솔루션 설계

고객경험 도메인 데이터 수집
- HTML, CSS, Web Crawling, 고객경험 데이터 수집/분석/ 서비스 기획 실습. 수집된 데이터 활용 방안 수립

디지털 맥락 파악 및 고객 감성분석
- 머신러닝, 텍스트마이닝, 시각화, 감정분석, Word2Vec, 키워드 추출

X+AI 비지니스 가치 설계
- 페르소나정의, ActionMap 작성, 기회영역 분석, 고객경험 디자인수행, AI기술, BMC, AI 윤리

AI활용 고객경험 분석
- 딥러닝, 컴퓨터 비전, 텍스트마이닝, 챗봇, 음성처리

도메인데이터 분석기반 MVP 설계
- 프로젝트관리, KPI/OKR, 사용자 피드백, UX, 서비스 블루프린트, PoC, 노코드 AI 도구, 프로토타입 제작

'''

In [50]:
doc4 = Document(
    page_content = text1,
    metadata = {'source' : 'https://smhrd.or.kr/course/aic_network/'}
)

In [52]:
# 4개의 문서를 하나의 리스트에 추가하여 웹 청크로 구성
web_chunk = [doc1, doc2, doc3, doc4]

In [53]:
web_chunk

[Document(metadata={'source': 'https://smhrd.or.kr/course/aic_network/'}, page_content='\n    6개월 과정960시간오프라인\n파이썬과 AI수학\n- 파이썬 분석/시각화, 선형대수, 확률, 통계\n\n컴퓨터 비전\n- 영상처리, 객체탐지/인식, 동작인식, 얼굴인식, 영역분할\n\n머신러닝/딥러닝\n- 지도/비지도학습, 평가지표, 신경망, CNN, RNN, Transformer, LMM/VLM/VLA\n\n로봇 운동학\n- 3차원공간 이해, 좌표계 변환, 자유도, FK/IK 해석, 로봇 움직임 제어, PID\n\nROS2과 SLAM\n- ROS2, 거리 및 IMU 센서, 점군 데이터 처리, SLAM, Navigation\n\n가상 로봇 설계\n- URDF/SDF기반 설계, 가상센서, 가상 로봇 제어\n\n가상 로봇 제어\n- 맵핑, 장애물 회피, 복구행동, 경로계획, 행동결정, 로봇팔, 다중로봇, 자율주행\n\n현실 로봇 제어\n- ROS brigde, 카메라/Lidar 연동, 모터 제어, 자율주행\n\n배포 서비스 구현\n- Streamlit/FastAPI\n\nAI 서비스 설계\n- AI 웹 서비스 기술 (HTML/CSS, JS, Node.js. React.js)\n\n'),
 Document(metadata={'source': 'https://smhrd.or.kr/course/aic_network/'}, page_content='\n    6개월 과정960시간오프라인\n파이썬과 AI수학\n- 파이썬 분석/시각화, 선형대수, 확률, 통계\n\n컴퓨터 비전\n- 영상처리, 객체탐지/인식, 동작인식, 얼굴인식, 영역분할\n\n머신러닝/딥러닝\n- 지도/비지도학습, 평가지표, 신경망, CNN, RNN, Transformer, LMM/VLM/VLA\n\n로봇 운동학\n- 3차원공간 이해, 좌표계 변환, 자유도, FK/IK 해석, 로봇 움직임 제어, PID\n\nROS2과 SLAM\n- ROS2,

In [54]:
# 임베딩 및 인덱싱
vec_db2 = FAISS.from_documents(
    documents = web_chunk,   # 임베딩할 벡터
    embedding = OpenAIEmbeddings()   # 임베딩 도구 연결
)

In [55]:
# 사용자 질문과 유사한 청크 2개를 찾는 검색기 생성
web_retriever = vec_db2.as_retriever(search_kwargs = {'k' : 1})

### 4. 교육과정 챗봇 구성하기
 - 사용자의 질문에 커리큘럼 내용이 있으면 web_retriever가 수행되도록 branch 구성

In [64]:
extract_info = ChatPromptTemplate.from_template('''
    아래 query 데이터에서 사용자가 과정에대한 질문을 하는지 파악하고
    과정 커리큘럼에 대한 질문이라면 특정 과정을 문의하는지 파악해서 알려줘.

    출력형식은 아래와 같이 JSON 형태로 만들어줘. JSON 파싱이 가능하도록 꼭 포멧 지켜줘.

    과정문의 : True, 과정타입 : 헬스케어

    만약 과정타입이 정보에 없다면 없음이라고 표기해.

    query : {query}
''')

In [65]:
# 체인구성
test_chain = {'query' : QT_chain} | extract_info | llm_4o_mini | JsonOutputParser()

In [66]:
test_chain.invoke('AI캠퍼스의 교육과정에는 어떤 과목들이 포함되어 있나요?')

{'과정문의': True, '과정타입': '없음'}

In [67]:
test_chain.invoke('헬스케어 교육듣고 싶어요')

{'과정문의': True, '과정타입': '헬스케어'}

In [68]:
test_chain.invoke('AI캠퍼스 사업이 무엇인가요?')

{'과정문의': False, '과정타입': '없음'}

### 구조 변경하기

In [73]:
extract_chain = extract_info | llm_4o_mini | JsonOutputParser()

In [74]:
test_chain2 = {'query' : QT_chain} | RunnableParallel(
    info = extract_chain,
    QT_result = RunnablePassthrough()
)

In [76]:
test_chain2.invoke('AI캠퍼스 사업이 뭐야?')

{'info': {'과정문의': False, '과정타입': '없음'},
 'QT_result': {'query': {'user_intent': 'AI 캠퍼스 사업의 개요와 목적에 대한 이해',
   'normalize_query': 'AI 캠퍼스 사업이 무엇인가요?',
   'expanded_query': ['AI 캠퍼스 사업의 주요 프로그램과 서비스는 무엇인가요?',
    'AI 캠퍼스 사업이 지향하는 목표와 비전은 무엇인가요?',
    'AI 캠퍼스 사업의 참여 혜택은 어떤 것들이 있나요?']}}}

In [109]:
# 브렌치 추가하기
web_chain = RunnableParallel(
    context = RunnableLambda(lambda x : x['QT_result']) | RunnableLambda( lambda x : PromptTemplate.from_template('{query}').invoke(x).text) | web_retriever,
    question = RunnableLambda(lambda x : x['QT_result']['query'])
)
pdf_chain = RunnableParallel(
    context = RunnableLambda(lambda x : x['QT_result'])| RunnableLambda( lambda x : PromptTemplate.from_template('{query}').invoke(x).text) | pdf_retriever,
    question = RunnableLambda(lambda x : x['QT_result']['query'])
)

# 챗봇용 템플릿 생성
chatbot_template = ChatPromptTemplate.from_messages([
    ('system','너는 교육과정 응답을 하는 챗봇이야. 아래 context 내용과 question을 기반으로 답변해줘'),
    ('system', 'context : {context}'),
    ('human', 'question : {question}')
])

final_chain = test_chain2 | RunnableBranch(
                      (lambda x : x['info']['과정문의'] == True, lambda x : web_chain),
                      lambda x : pdf_chain
              ) | chatbot_template | llm_4o_mini | StrOutputParser()

In [110]:
final_chain.invoke('AI캠퍼스 사업이 뭐야?')

"AI 캠퍼스는 AI 산업의 인력 수요와 국내외 AI 직무 분류를 고려하여, 'AI 엔지니어', 'AI 어플리케이션 개발자', 'AI 융합가', 'AI 하드웨어 엔지니어' 등 4개 직군의 실무 인력을 양성하는 것을 목표로 하고 있습니다. 이 사업에 참여하고자 하는 기관은 이러한 인력 양성 목표에 맞춰 훈련 과정을 설계해야 하며, 기업의 실제 문제를 반영한 프로젝트 학습 비중을 최소 30% 이상 편성해야 합니다.\n\nAI 캠퍼스는 실질적으로 AI 기술을 활용한 다양한 직무에 필요한 교육 및 훈련 프로그램을 제공하며, AI 시스템 구축 및 운영, 고성능 AI 모델 연계 등 구체적인 업무 수행 능력을 배양하는 데 중점을 둡니다. 더불어, AI 캠퍼스는 AI 기술의 발전과 함께 산업 전반에 긍정적인 영향을 미치고, 인력 시장에서의 경쟁력을 높이는 데 기여할 것으로 기대됩니다. \n\n추후 전망에 관해서는, AI 캠퍼스가 제공하는 교육 프로그램을 통해 인력 양성이 이루어지면, AI 산업의 성장과 고용 창출은 더욱 가속화될 것으로 보이며, 이는 AI 기술의 활용이 증가하는 추세와 맞물려 긍정적인 시장 영향을 미칠 것입니다."

### Colab에서 외부로 Langchain 연결하기

### 학습목표
- MVP를 구성할때 colab 환경에서 langchain을 구성하고 외부에서 호출 가능하게 만들 수 있다
- 실제 프로덕션 단계에서는 사용이 불가능, MVP 시연용으로 활용
- 또 GPU 환경으로 모델을 배포하는 경우에도 응용가능

### 패키지 설치
- langchain, langchain-openai : 랭체인 구성을 위한 패키지
- langserve : 랭체인으로 구성한 체인을 REST API로 감싸는 패키지
- fastapi, uvicorn : 파이썬 서버를 구성하기 위한 도구
- pyngrok : 터널링을 이용해 외부통신이 가능하게 해주는 도구

In [112]:
!pip install langserve fastapi uvicorn pyngrok

#### FastAPI + LangServe 붙이기

In [113]:
from fastapi import FastAPI
from langserve import add_routes

In [114]:
# 서버생성
app = FastAPI()

add_routes(app,   # 라우팅을 추가할 서버
           final_chain,   # 라우팅에 연결할 chain
           path = '/query'   # 해당 체인을 접속할 URL path
           )

### Colab에서 서버 실행

In [115]:
import nest_asyncio
import uvicorn
from threading import Thread

In [116]:
# 노트북 커널과 별개로 수행이 가능하도록 셀 셋팅
nest_asyncio.apply()

def run() :
    uvicorn.run(app,   # 구동할 서버앱
                host = '0.0.0.0',   # 노트북이 돌고있는 pc의 ip
                port = 8000   # 통신이 가능한 포트(출입구)
                )

thread = Thread(target = run)   # 쓰레드 생성
thread.start()   # 별도의 쓰레드 구동

### colab 인스턴스 내부에서 호출하기

In [127]:
!curl -X POST 'http://0.0.0.0:8000/query/invoke' \
-H 'Content-Type: application/json' \
-d '{"input" : {"query" : "AI캠퍼스 사업이 뭐야?"}}'

INFO:     127.0.0.1:52574 - "POST /query/invoke HTTP/1.1" 200 OK
{"output":"AI 캠퍼스 사업은 AI 산업의 인력 수요와 관련된 직무 분류를 고려하여, 다양한 AI 관련 직군의 실무 인력을 양성하는 것을 목표로 합니다. 이 사업의 주요 직군은 'AI 엔지니어', 'AI 어플리케이션 개발자', 'AI 융합가', 'AI 하드웨어 엔지니어' 등 네 가지로 나뉘어 있습니다. 참여기관은 이러한 목표에 맞춰 훈련 과정을 설계하고, 기업의 현업 문제를 반영한 프로젝트 학습의 비중을 30% 이상 편성해야 합니다.\n\nAI 캠퍼스에서 제공하는 프로그램은 각 직군에 맞는 이론 교육과 실습, 프로젝트 학습 등이 포함되며, 실질적인 산업 문제 해결 능력을 기르는 데 중점을 둡니다.\n\nAI 캠퍼스 사업의 장점은 산업 현장에서 필요로 하는 기술 인력을 빠르게 양성할 수 있다는 점이며, 이는 AI 기술 발전에 기여하고, 기업의 경쟁력을 높이는 효과를 기대할 수 있습니다. 또한, 실제 사용자와의 연계를 통해 실무 감각을 익힐 수 있어, 졸업 후 취업에 유리한 조건을 갖추게 됩니다.","metadata":{"run_id":"6a6e3757-a17e-40d7-a95d-a7b63002d95c","feedback_tokens":[]}}

### ngrok 터널 열기
 - 터널링을 통해 외부통신이 가능하도록 지원하는 서비스

In [ ]:
# 토큰등록
from pyngrok import ngrok
ngrok.set_auth_token('Authtoken_KEY')

In [129]:
public_url = ngrok.connect(8000)   # 현재 PC에서 서버가 돌고있는 포트 연결

In [131]:
# 무료 플랜이어서 URL은 변동 됨
public_url

<NgrokTunnel: "https://refresh-unstitch-onstage.ngrok-free.dev" -> "http://localhost:8000">